# 03b · Supervised models: card-settings run, imbalance experiment, tuning log

Owner: Ayo (Y8–Y10 on v0, Y14 on v1). Random Forest and XGBoost from `src/models_supervised.py` on `data/processed/features_<DATASET>.parquet`. They train on **all** training rows (TEAM_PLAN 5.4) and are judged on the **val** split only; test is never looked at while tuning. Every run adds one row to `results/tables/tuning_log_supervised.csv`: parameters, imbalance set-up, val ROC-AUC, val PR-AUC, val recall at the top 1% of val user-days, and training time. The best run has the highest **val PR-AUC**.

Y14 order: card-settings run → imbalance experiment → each model's set-up (`ms.FINAL_SETUP`, DEC-Y) → the grids with that set-up → T4.

Run with Restart & Run All. A full run replaces only this dataset's rows in the tuning log and in `imbalance.csv`; the v0 history stays.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import models_supervised as ms
from src.config import PROCESSED_DIR, TABLES_DIR

DATASET = 'v1'
df = pd.read_parquet(PROCESSED_DIR / f'features_{DATASET}.parquet')
log_path = TABLES_DIR / ms.TUNING_LOG
ms.clear_log(DATASET)                  # drops only this dataset's runs; other datasets' history stays
X, y, cols = ms.get_model_matrix(df, split='train')
print(f'{DATASET}: {len(df):,} user-days, {len(cols)} features, {len(X):,} training rows '
      f'({int(y.sum())} malicious)')
df.groupby('split').label.agg(['size', 'sum']).rename(columns={'size': 'rows', 'sum': 'malicious'})

v1: 330,268 user-days, 76 features, 209,072 training rows (578 malicious)


,rows,malicious
split,,
test,65007,170
train,209072,578
val,56189,236


## Y8: first run with the card settings

Saved as `results/scores/<model>_<DATASET>.parquet` (the final `<model>.parquet` files come from the `final` command in Y15).

In [2]:
first = []
for name in ms.MODELS:
    log, _ = ms.tune_supervised(df, name, grid={}, dataset=DATASET)    # an empty grid = the card settings
    scores, info = ms.train_and_score_supervised(df, name, save_as=f'{name}_{DATASET}')
    first.append(log.assign(saved=str(info['path'].relative_to(ROOT))))
pd.concat(first)[['model', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'best_iteration', 'fit_seconds', 'saved']].round(4)

,model,val_roc_auc,val_pr_auc,val_recall_top1pct,best_iteration,fit_seconds,saved
0,random_forest,0.9998,0.9584,0.9958,None,15.13,results/scores/random_forest_v1.parquet
0,xgboost,0.9999,0.9906,1.0000,359,6.38,results/scores/xgboost_v1.parquet


## Y10: imbalance experiment

Three set-ups per model, all with the card settings: (a) no weighting, (b) class weights (`class_weight='balanced'` / `scale_pos_weight`), (c) SMOTE to a 10% minority **on the training rows only** (inside an imblearn Pipeline, so val and test are never resampled). Saved in `results/tables/imbalance.csv` (a rerun replaces only this dataset's rows).

In [3]:
imbalance = ms.imbalance_experiment(df, dataset=DATASET)
imbalance.round(4)

,model,setup,dataset,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
0,random_forest,a_no_weighting,v1,0.9995,0.9398,0.9873,22.38
1,random_forest,b_class_weights,v1,0.9998,0.9584,0.9958,11.60
2,random_forest,c_smote_train_only,v1,0.9999,0.9767,0.9958,35.66
3,xgboost,a_no_weighting,v1,0.9999,0.9854,1.0000,9.52
4,xgboost,b_class_weights,v1,0.9999,0.9906,1.0000,6.72
5,xgboost,c_smote_train_only,v1,0.9999,0.9901,1.0000,9.52


## Each model's set-up (DEC-Y)

Card rule: the set-up with the highest val PR-AUC above. The choice itself is Ayo's and lives in `ms.FINAL_SETUP`; this cell shows it next to the rule's pick.

In [4]:
rule = imbalance.loc[imbalance.groupby('model')['val_pr_auc'].idxmax()].set_index('model')['setup']
pd.DataFrame({'card_rule': rule, 'FINAL_SETUP': pd.Series(ms.FINAL_SETUP)})

,card_rule,FINAL_SETUP
random_forest,c_smote_train_only,c_smote_train_only
xgboost,b_class_weights,b_class_weights


## Y9: grid search with each model's set-up

- **Random Forest:** `n_estimators` {200, 500} × `max_depth` {None, 10, 20} × `min_samples_leaf` {1, 5} × `max_features` {sqrt, 0.5} = 24 runs.
- **XGBoost:** `max_depth` {3, 6} × `learning_rate` {0.05, 0.1} × `subsample` {0.8, 1.0} × `min_child_weight` {1, 5} = 16 runs; the number of trees comes from early stopping on val.

`tune_final()` also writes the best settings per model to `results/tables/best_params_supervised.json`.

In [5]:
best = ms.tune_final(df, dataset=DATASET)
runs = ms.read_log()
runs = runs[(runs['dataset'] == DATASET) & (runs['params'] != '{}')]
for name in ms.MODELS:
    print(name, best[name]['setup'], best[name]['params'])
runs.sort_values('val_pr_auc', ascending=False).groupby('model').head(5)[
    ['model', 'setup', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds']].round(4)

random_forest c_smote_train_only {'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 500}
xgboost b_class_weights {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 1.0}


,model,setup,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
79,xgboost,b_class_weights,"{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",1.0000,0.9913,1.0000,9.36
80,xgboost,b_class_weights,"{""learning_rate"": 0.1, ""max_depth"": 6, ""min_ch...",1.0000,0.9911,1.0000,5.35
77,xgboost,b_class_weights,"{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",0.9999,0.9908,0.9958,10.99
73,xgboost,b_class_weights,"{""learning_rate"": 0.05, ""max_depth"": 6, ""min_c...",0.9999,0.9907,1.0000,10.02
72,xgboost,b_class_weights,"{""learning_rate"": 0.05, ""max_depth"": 6, ""min_c...",0.9999,0.9906,1.0000,10.11
67,random_forest,c_smote_train_only,"{""max_depth"": 20, ""max_features"": 0.5, ""min_sa...",0.9999,0.9837,0.9958,339.44
66,random_forest,c_smote_train_only,"{""max_depth"": 20, ""max_features"": 0.5, ""min_sa...",0.9999,0.9837,0.9958,132.51
51,random_forest,c_smote_train_only,"{""max_depth"": null, ""max_features"": 0.5, ""min_...",0.9999,0.9836,0.9958,297.63
50,random_forest,c_smote_train_only,"{""max_depth"": null, ""max_features"": 0.5, ""min_...",0.9999,0.9832,0.9958,123.16
65,random_forest,c_smote_train_only,"{""max_depth"": 20, ""max_features"": 0.5, ""min_sa...",0.9999,0.9799,1.0000,325.17


## The tuning log and table T4

In [6]:
log = ms.read_log()
print(f'{len(log)} runs in {log_path.relative_to(ROOT)}:', log.groupby(['dataset', 'setup']).size().to_dict())
ms.hyperparameter_table(dataset=DATASET).round(4)

84 runs in results/tables/tuning_log_supervised.csv: {('v0', 'b_class_weights'): 42, ('v1', 'b_class_weights'): 18, ('v1', 'c_smote_train_only'): 24}


,model,defaults,grid,best_params,dataset,setup,val_pr_auc,val_roc_auc,runs
0,random_forest,"{""n_estimators"": 300, ""class_weight"": ""balanced""}","{""n_estimators"": [200, 500], ""max_depth"": [nul...","{""max_depth"": 20, ""max_features"": 0.5, ""min_sa...",v1,c_smote_train_only,0.9837,0.9999,25
1,xgboost,"{""n_estimators"": 1000, ""max_depth"": 6, ""learni...","{""max_depth"": [3, 6], ""learning_rate"": [0.05, ...","{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",v1,b_class_weights,0.9913,1.0000,17


**Notes (Ayo):** what the v0 numbers say, which imbalance set-up to keep for the final models, and which settings to carry to v1. *Human-only: write your own reading here.*